# Week 78, Day 1: Command-line apps

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w078d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import argparse

parser = argparse.ArgumentParser(prog="greet")
parser.add_argument("name")
parser.add_argument("--times", type=int, default=1)
parser.add_argument("--shout", action="store_true")

for argv in (["Ada"], ["Ada", "--times", "2"], ["--shout", "Grace"]):
    args = parser.parse_args(argv)
    for _ in range(args.times):
        text = f"Hello, {args.name}!"
        print(text.upper() if args.shout else text)

In [ ]:
import argparse, pathlib

def positive(text):
    value = int(text)
    if value <= 0:
        raise argparse.ArgumentTypeError(f"{text} is not a positive whole number")
    return value

parser = argparse.ArgumentParser(prog="resize", description="Resize images.")
parser.add_argument("files", nargs="+", type=pathlib.Path, help="images to resize")
parser.add_argument("--width", type=positive, default=800, help="new width in pixels")
parser.add_argument("--format", choices=["png", "jpg", "webp"], default="png")
parser.add_argument("-v", "--verbose", action="count", default=0)
size = parser.add_mutually_exclusive_group()
size.add_argument("--keep-ratio", action="store_true")
size.add_argument("--height", type=positive)

args = parser.parse_args(["cat.png", "dog.png", "--width", "200", "-vv", "--keep-ratio"])
print(args.files, args.width, args.format, args.verbose, args.keep_ratio, args.height)

In [ ]:
import argparse

def cmd_add(args):
    print(f"added {args.title!r} with priority {args.priority}")

def cmd_list(args):
    print("listing", "all" if args.all else "open", "tasks")

parser = argparse.ArgumentParser(prog="todo")
sub = parser.add_subparsers(dest="command", required=True)

p_add = sub.add_parser("add", help="add a task")
p_add.add_argument("title")
p_add.add_argument("-p", "--priority", type=int, choices=range(1, 4), default=2)
p_add.set_defaults(func=cmd_add)

p_list = sub.add_parser("list", help="list tasks")
p_list.add_argument("--all", action="store_true")
p_list.set_defaults(func=cmd_list)

for argv in (["add", "Write report", "-p", "1"], ["list", "--all"]):
    args = parser.parse_args(argv)
    args.func(args)

In [ ]:
import argparse, io, sys

class UsageError(Exception):
    pass

class Parser(argparse.ArgumentParser):
    def error(self, message):
        raise UsageError(message)

def main(argv, out=sys.stdout, err=sys.stderr):
    parser = Parser(prog="half")
    parser.add_argument("number", type=int)
    try:
        args = parser.parse_args(argv)
    except UsageError as e:
        print(f"half: {e}", file=err)
        return 2
    if args.number % 2:
        print(f"half: {args.number} is odd", file=err)
        return 1
    print(args.number // 2, file=out)
    return 0

for argv in (["10"], ["7"], ["ten"]):
    out, err = io.StringIO(), io.StringIO()
    code = main(argv, out, err)
    print(argv, "exit", code, "out", repr(out.getvalue()), "err", repr(err.getvalue()))

In [ ]:
import tomllib

DEFAULTS = {"data": "tasks.json", "colour": True, "width": 80}
config = tomllib.loads('''
width = 100
colour = false
''')
env = {"TODO_DATA": "/tmp/work.json"}
flags = {"data": None, "width": 120, "colour": None}   # None: not given

settings = dict(DEFAULTS)
settings.update(config)
settings.update({k[5:].lower(): v for k, v in env.items() if k.startswith("TODO_")})
settings.update({k: v for k, v in flags.items() if v is not None})
print(settings)

> **Key idea.** argparse describes a command line: positionals, options with type, choices, default, required, nargs, store_true, count and append actions, mutually exclusive groups, and help for everything; a type function that raises ArgumentTypeError gives a clean usage error. Subcommands use add_subparsers(dest=..., required=True), one parser each, and set_defaults(func=...) so main calls args.func(args). Results go to stdout, messages to stderr; exit 0 for success, 1 for failure, 2 for usage errors (argparse prints usage to stderr and exits 2; catch SystemExit or override error to control it). Settings merge in order defaults, config file, environment, flags, each overriding only what it sets (default=None tells "not given" apart). Write main(argv) that returns the exit code, take streams, environment and clock as parameters, keep the work in plain functions, and add --dry-run and --yes for anything destructive.

### Exercise: Describe a command line

Write `build_parser()` returning an `ArgumentParser` with `prog="backup"` for this tool: one or more `sources` (positional, as `pathlib.Path`); `-d`/`--dest` (a `Path`, required); `--keep` (an `int`, default `7`); `--compress` (one of `"none"`, `"zip"`, `"gzip"`, default `"zip"`); `-n`/`--dry-run` (a flag); `-v`/`--verbose` (a counter starting at `0`); and `--exclude` (repeatable, collecting every pattern in a list, with `default=[]`).

In [ ]:
import argparse
import pathlib

def build_parser():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parser():\n    \"every kind of argument\"\n    import argparse, io, contextlib, pathlib\n    p = build_parser()\n    assert p.prog == \"backup\"\n    a = p.parse_args([\"docs\", \"photos\", \"-d\", \"/mnt/usb\"])\n    assert a.sources == [pathlib.Path(\"docs\"), pathlib.Path(\"photos\")] and a.dest == pathlib.Path(\"/mnt/usb\")\n    assert a.keep == 7 and a.compress == \"zip\" and a.dry_run is False and a.verbose == 0 and a.exclude == []\n    b = build_parser().parse_args([\"src\", \"--dest\", \"out\", \"--keep\", \"3\", \"--compress\", \"gzip\", \"-n\", \"-vv\", \"--exclude\", \"*.tmp\", \"--exclude\", \"cache\"])\n    assert b.keep == 3 and b.compress == \"gzip\" and b.dry_run is True and b.verbose == 2 and b.exclude == [\"*.tmp\", \"cache\"]\n\ndef test_mistakes():\n    \"usage errors exit with 2\"\n    import io, contextlib\n    for argv in ([\"src\"], [\"-d\", \"out\"], [\"src\", \"-d\", \"out\", \"--keep\", \"many\"], [\"src\", \"-d\", \"out\", \"--compress\", \"rar\"]):\n        with contextlib.redirect_stderr(io.StringIO()):\n            try:\n                build_parser().parse_args(argv)\n            except SystemExit as e:\n                assert e.code == 2, argv\n            else:\n                raise AssertionError(f\"{argv} should be a usage error\")")

### Exercise: Types that validate

Write three argparse type functions. `port(text)`: an `int` from `1` to `65535`, else raise `argparse.ArgumentTypeError(f"{text} is not a port (1 to 65535)")`. `duration(text)`: seconds from text like `"90s"`, `"15m"`, `"2h"` or `"1d"` (a whole number then one unit), else raise `ArgumentTypeError(f"{text} is not a duration like 15m")`. `existing_dir(text)`: a `pathlib.Path` that is an existing directory, else raise `ArgumentTypeError(f"no such folder: {text}")`. Then `build_parser()` with `prog="serve"`, a `--port` (type `port`, default `8000`), an `--every` (type `duration`, default `"1h"`, which argparse converts too) and a positional `root` (type `existing_dir`).

In [ ]:
import argparse
import pathlib
import re

def port(text):
    ___

def duration(text):
    ___

def existing_dir(text):
    ___

def build_parser():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_types():\n    \"each type\"\n    import argparse, pathlib\n    assert port(\"8080\") == 8080 and port(\"1\") == 1 and port(\"65535\") == 65535\n    for bad in (\"0\", \"70000\", \"http\"):\n        try:\n            port(bad)\n        except argparse.ArgumentTypeError as e:\n            assert \"is not a port\" in str(e)\n        else:\n            raise AssertionError(bad)\n    assert duration(\"90s\") == 90 and duration(\"15m\") == 900 and duration(\"2h\") == 7200 and duration(\"1d\") == 86400\n    for bad in (\"15\", \"m\", \"1.5h\", \"2w\", \"-3m\"):\n        try:\n            duration(bad)\n        except argparse.ArgumentTypeError as e:\n            assert str(e) == f\"{bad} is not a duration like 15m\"\n        else:\n            raise AssertionError(bad)\n    pathlib.Path(\"site\").mkdir(exist_ok=True)\n    assert existing_dir(\"site\") == pathlib.Path(\"site\")\n    try:\n        existing_dir(\"nowhere\")\n    except argparse.ArgumentTypeError as e:\n        assert str(e) == \"no such folder: nowhere\"\n    else:\n        raise AssertionError(\"nowhere\")\n\ndef test_parser():\n    \"argparse uses them, defaults included\"\n    import io, contextlib, pathlib\n    pathlib.Path(\"site\").mkdir(exist_ok=True)\n    a = build_parser().parse_args([\"site\"])\n    assert a.port == 8000 and a.every == 3600 and a.root == pathlib.Path(\"site\")\n    b = build_parser().parse_args([\"site\", \"--port\", \"9000\", \"--every\", \"30m\"])\n    assert b.port == 9000 and b.every == 1800\n    err = io.StringIO()\n    with contextlib.redirect_stderr(err):\n        try:\n            build_parser().parse_args([\"site\", \"--port\", \"99999\"])\n        except SystemExit as e:\n            assert e.code == 2\n    assert \"99999 is not a port\" in err.getvalue()")

### Exercise: A tool with subcommands

Write `main(argv, tasks, out)` for a `todo` tool (`prog="todo"`): `tasks` is a list of dicts (`{"title": ..., "done": False, "priority": 2}`) that the commands change, and output goes to `out` with `print(..., file=out)`. Subcommands (required): `add TITLE [-p {1,2,3}]` appends a task and prints `added #N: TITLE` (N counts from 1); `list [--all]` prints `#N [ ] TITLE (pN)` for each open task (or `[x]` for done ones, which are only shown with `--all`), or `nothing to do` if nothing is shown; `done N` marks task N done and prints `done #N: TITLE`. Use `set_defaults(func=...)` and return the function's result (`0`) from `main`.

In [ ]:
import argparse

def main(argv, tasks, out):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_todo():\n    \"add, list and done\"\n    import io\n    tasks, out = [], io.StringIO()\n    assert main([\"list\"], tasks, out) == 0\n    assert main([\"add\", \"Write report\", \"-p\", \"1\"], tasks, out) == 0\n    assert main([\"add\", \"Water plants\"], tasks, out) == 0\n    assert main([\"done\", \"2\"], tasks, out) == 0\n    assert main([\"list\"], tasks, out) == 0\n    assert main([\"list\", \"--all\"], tasks, out) == 0\n    assert out.getvalue().splitlines() == [\n        \"nothing to do\", \"added #1: Write report\", \"added #2: Water plants\", \"done #2: Water plants\",\n        \"#1 [ ] Write report (p1)\", \"#1 [ ] Write report (p1)\", \"#2 [x] Water plants (p2)\"]\n    assert tasks[1] == {\"title\": \"Water plants\", \"done\": True, \"priority\": 2}\n\ndef test_usage():\n    \"a missing subcommand or a bad priority is a usage error\"\n    import io, contextlib\n    for argv in ([], [\"add\", \"x\", \"-p\", \"5\"], [\"fly\"]):\n        with contextlib.redirect_stderr(io.StringIO()):\n            try:\n                main(argv, [], io.StringIO())\n            except SystemExit as e:\n                assert e.code == 2\n            else:\n                raise AssertionError(argv)")

### Exercise: Exit codes and two streams

Write `main(argv, out, err)` for `wc-lines` (`prog="wc-lines"`): it takes one or more file paths and prints `COUNT PATH` to `out` for each readable file (the number of lines), then `COUNT total` if there was more than one file. A missing file prints `wc-lines: PATH: no such file` to `err` and is skipped, and the exit code is then `1` (the other files are still counted). A usage mistake (no paths at all) prints `wc-lines: ` followed by argparse's message to `err` and returns `2`: override `error` in a parser subclass so nothing is printed by argparse itself and nothing exits. Success returns `0`.

In [ ]:
import argparse

class UsageError(Exception):
    pass

def main(argv, out, err):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_counts():\n    \"results on out, problems on err\"\n    import io\n    write_file(\"a.txt\", \"one\\ntwo\\nthree\\n\")\n    write_file(\"b.txt\", \"only\\n\")\n    out, err = io.StringIO(), io.StringIO()\n    assert main([\"a.txt\", \"b.txt\"], out, err) == 0\n    assert out.getvalue() == \"3 a.txt\\n1 b.txt\\n4 total\\n\" and err.getvalue() == \"\"\n    out, err = io.StringIO(), io.StringIO()\n    assert main([\"a.txt\"], out, err) == 0 and out.getvalue() == \"3 a.txt\\n\"\n\ndef test_failures():\n    \"1 for a missing file, 2 for usage\"\n    import io\n    write_file(\"a.txt\", \"one\\ntwo\\nthree\\n\")\n    out, err = io.StringIO(), io.StringIO()\n    assert main([\"a.txt\", \"nope.txt\"], out, err) == 1\n    assert out.getvalue() == \"3 a.txt\\n3 total\\n\" and err.getvalue() == \"wc-lines: nope.txt: no such file\\n\"\n    out, err = io.StringIO(), io.StringIO()\n    assert main([], out, err) == 2\n    assert out.getvalue() == \"\" and err.getvalue().startswith(\"wc-lines: \") and \"required\" in err.getvalue()")

### Exercise: Settings from four places

Write `load_settings(argv, env, config_text)`. Defaults: `{"data": "tasks.json", "width": 80, "colour": True}`. `config_text` is a TOML document (possibly empty) whose keys override the defaults. Then environment variables `TODO_DATA`, `TODO_WIDTH` (an `int`) and `TODO_COLOUR` (`"1"`, `"true"`, `"yes"` or `"on"` mean `True`; `"0"`, `"false"`, `"no"` or `"off"` mean `False`, ignoring case; anything else raises `ValueError(f"TODO_COLOUR must be true or false, not {value!r}")`). Then flags parsed from `argv` (`prog="todo"`): `--data`, `--width` (an `int`) and `--colour`/`--no-colour` (use `action=argparse.BooleanOptionalAction`), all with `default=None` so only flags actually given override. Return `(settings, sources)`, where `sources` maps each key to where its value came from: `"default"`, `"config"`, `"env"` or `"flag"`.

In [ ]:
import argparse
import tomllib

DEFAULTS = {"data": "tasks.json", "width": 80, "colour": True}

def load_settings(argv, env, config_text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_order():\n    \"later sources win\"\n    s, src = load_settings([], {}, \"\")\n    assert s == {\"data\": \"tasks.json\", \"width\": 80, \"colour\": True} and set(src.values()) == {\"default\"}\n    s, src = load_settings([\"--width\", \"120\"], {\"TODO_DATA\": \"/tmp/w.json\", \"TODO_WIDTH\": \"100\"}, 'width = 90\\ncolour = false\\n')\n    assert s == {\"data\": \"/tmp/w.json\", \"width\": 120, \"colour\": False}\n    assert src == {\"data\": \"env\", \"width\": \"flag\", \"colour\": \"config\"}\n    s, src = load_settings([\"--colour\"], {\"TODO_COLOUR\": \"Off\"}, \"\")\n    assert s[\"colour\"] is True and src[\"colour\"] == \"flag\"\n    s, src = load_settings([], {\"TODO_COLOUR\": \"Off\"}, \"\")\n    assert s[\"colour\"] is False and src[\"colour\"] == \"env\"\n    s, src = load_settings([\"--no-colour\"], {}, \"\")\n    assert s[\"colour\"] is False\n\ndef test_bad_env():\n    \"a clear error for a bad value\"\n    try:\n        load_settings([], {\"TODO_COLOUR\": \"maybe\"}, \"\")\n    except ValueError as e:\n        assert str(e) == \"TODO_COLOUR must be true or false, not 'maybe'\"\n    else:\n        raise AssertionError(\"expected ValueError\")")

### Exercise: Dry runs and confirmation

Write `clean(argv, files, ask, out)` for a tool that deletes files: `files` is a set of file names standing in for a folder, and `ask(prompt)` returns the person's answer (like `input`). Arguments (`prog="clean"`): one or more `patterns` (matched with `fnmatch.fnmatch`), `-n`/`--dry-run` and `-y`/`--yes`. Find the matching files in sorted order. If there are none, print `nothing to delete` and return `0`. With `--dry-run`, print `would delete NAME` for each and return `0` without asking. Otherwise, unless `--yes` was given, ask `Delete N file(s)? [y/N] `: anything except `y` or `yes` (ignoring case and spaces) prints `cancelled` and returns `1`. Then remove each from `files`, print `deleted NAME`, and return `0`.

In [ ]:
import argparse
import fnmatch

def clean(argv, files, ask, out):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_clean():\n    \"dry run, confirmation and --yes\"\n    import io\n    asked = []\n    def ask(prompt, answer):\n        asked.append(prompt)\n        return answer\n    files = {\"a.tmp\", \"b.log\", \"c.tmp\", \"notes.txt\"}\n    out = io.StringIO()\n    assert clean([\"*.tmp\", \"-n\"], files, lambda p: ask(p, \"y\"), out) == 0\n    assert out.getvalue() == \"would delete a.tmp\\nwould delete c.tmp\\n\" and asked == [] and len(files) == 4\n    out = io.StringIO()\n    assert clean([\"*.tmp\"], files, lambda p: ask(p, \"n\"), out) == 1\n    assert out.getvalue() == \"cancelled\\n\" and asked == [\"Delete 2 file(s)? [y/N] \"] and len(files) == 4\n    out = io.StringIO()\n    assert clean([\"*.tmp\", \"*.log\"], files, lambda p: ask(p, \" Yes \"), out) == 0\n    assert out.getvalue() == \"deleted a.tmp\\ndeleted b.log\\ndeleted c.tmp\\n\" and files == {\"notes.txt\"}\n    out = io.StringIO()\n    assert clean([\"*.txt\", \"-y\"], files, lambda p: ask(p, \"no\"), out) == 0\n    assert files == set() and len(asked) == 2\n    out = io.StringIO()\n    assert clean([\"*.png\"], files, lambda p: ask(p, \"y\"), out) == 0 and out.getvalue() == \"nothing to delete\\n\"")

### Bug hunt: Adding strings

The tool should add its numbers, but they arrive as strings, so `"2" + "3"`-style joining makes this print `234`. Tell argparse to convert them, so it prints `9`.

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog="add")
parser.add_argument("numbers", nargs="+")
args = parser.parse_args(["2", "3", "4"])
total = args.numbers[0]
for n in args.numbers[1:]:
    total += n
print(total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"type=int\"\n    expect_output(\"9\")")

### Bug hunt: A flag that was always on

`--verbose` was written with `type=bool`, but `bool("False")` is `True` (any non-empty string is), so asking for quiet output still prints `verbose: True`. Make it a proper flag (off unless given, and taking no value), and update the call so it prints `verbose: False` then `verbose: True`.

In [ ]:
import argparse

parser = argparse.ArgumentParser(prog="sync")
parser.add_argument("--verbose", type=bool, default=False)
print("verbose:", parser.parse_args(["--verbose", "False"]).verbose)
print("verbose:", parser.parse_args(["--verbose", "True"]).verbose)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"store_true\"\n    expect_output(\"verbose: False\\nverbose: True\")\n    assert \"store_true\" in source and \"type=bool\" not in source, \"use action=\\\"store_true\\\"\"")

### Bug hunt: An error that looked like data

This `lookup` tool prints its error to stdout and returns `0`, so a script piping its output treats `error: no user 42` as a real result, and this prints `out='error: no user 42\n' err='' code=0`. Print errors to `err` and return `1`, so it prints `out='' err='lookup: no user 42\n' code=1`.

In [ ]:
import io

USERS = {7: "ada"}

def lookup(user_id, out, err):
    if user_id not in USERS:
        print(f"error: no user {user_id}", file=out)
        return 0
    print(USERS[user_id], file=out)
    return 0

out, err = io.StringIO(), io.StringIO()
code = lookup(42, out, err)
print(f"out={out.getvalue()!r} err={err.getvalue()!r} code={code}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"stderr and exit 1\"\n    expect_output(\"out='' err='lookup: no user 42\\\\n' code=1\")")

### Puzzle: A testable entry point

Arrange a `main` that parses its arguments, reports usage errors itself, and returns an exit code, followed by the standard entry point. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the exit code argparse uses for a usage error
q = ___   # args.verbose after parsing -vvv with action="count", default=0
r = ___   # the attribute that --dry-run is stored under
s = ___   # "stdout" or "stderr": where a progress message should go
t = ___   # which wins, "env" or "flag", when both set the same setting

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 2 and q == 3 and r == \"dry_run\", \"p, q and r\"\n    assert s == \"stderr\" and t == \"flag\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a notes tool

Write `main(argv, store, out, err, env)` for `notes` (`prog="notes"`), where `store` is a dict from note id (an `int`, counting from 1) to a dict `{"text": ..., "tags": [...]}`. A main option `--format {text,json}` (before the subcommand) defaults to the environment variable `NOTES_FORMAT` if it's set, else `"text"`. Subcommands (required): `add TEXT [--tag TAG ...]` (repeatable `--tag`) stores the note with the next id (one more than the largest, or 1) and prints `added N`; `list [--tag TAG]` prints the notes (all, or those with that tag) in id order: as text, one line each, `N: TEXT [tag1, tag2]` (or `N: TEXT` without tags), or `no notes` if none match; as JSON, one line, `json.dumps` of a list of `{"id": N, "text": ..., "tags": [...]}`; `delete N` removes it and prints `deleted N`, or prints `notes: no note N` to `err` and returns `1`. Usage errors print `notes: MESSAGE` to `err` and return `2` (override `error`; nothing may exit). An unknown `NOTES_FORMAT` is also a usage error with the message `NOTES_FORMAT must be text or json`. Return `0` on success.

In [ ]:
import argparse
import json

class UsageError(Exception):
    pass

def main(argv, store, out, err, env):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_notes():\n    \"add, list, delete\"\n    import io\n    store, out, err = {}, io.StringIO(), io.StringIO()\n    assert main([\"list\"], store, out, err, {}) == 0\n    assert main([\"add\", \"Buy milk\", \"--tag\", \"home\", \"--tag\", \"shop\"], store, out, err, {}) == 0\n    assert main([\"add\", \"Call Ada\"], store, out, err, {}) == 0\n    assert main([\"list\"], store, out, err, {}) == 0\n    assert main([\"list\", \"--tag\", \"shop\"], store, out, err, {}) == 0\n    assert main([\"delete\", \"1\"], store, out, err, {}) == 0\n    assert main([\"add\", \"Water plants\"], store, out, err, {}) == 0\n    assert out.getvalue().splitlines() == [\"no notes\", \"added 1\", \"added 2\", \"1: Buy milk [home, shop]\", \"2: Call Ada\",\n                                           \"1: Buy milk [home, shop]\", \"deleted 1\", \"added 3\"]\n    assert err.getvalue() == \"\" and sorted(store) == [2, 3]\n\ndef test_json():\n    \"JSON from a flag or the environment\"\n    import io, json\n    store = {4: {\"text\": \"x\", \"tags\": [\"a\"]}, 2: {\"text\": \"y\", \"tags\": []}}\n    out = io.StringIO()\n    assert main([\"--format\", \"json\", \"list\"], store, out, io.StringIO(), {}) == 0\n    assert json.loads(out.getvalue()) == [{\"id\": 2, \"text\": \"y\", \"tags\": []}, {\"id\": 4, \"text\": \"x\", \"tags\": [\"a\"]}]\n    assert out.getvalue().count(\"\\n\") == 1\n    out = io.StringIO()\n    assert main([\"list\", \"--tag\", \"a\"], store, out, io.StringIO(), {\"NOTES_FORMAT\": \"json\"}) == 0\n    assert json.loads(out.getvalue()) == [{\"id\": 4, \"text\": \"x\", \"tags\": [\"a\"]}]\n    out = io.StringIO()\n    assert main([\"--format\", \"text\", \"list\"], store, out, io.StringIO(), {\"NOTES_FORMAT\": \"json\"}) == 0\n    assert out.getvalue() == \"2: y\\n4: x [a]\\n\"\n\ndef test_errors():\n    \"1 for a missing note, 2 for usage, and nothing exits\"\n    import io\n    out, err = io.StringIO(), io.StringIO()\n    assert main([\"delete\", \"9\"], {}, out, err, {}) == 1 and err.getvalue() == \"notes: no note 9\\n\" and out.getvalue() == \"\"\n    for argv, env in (([], {}), ([\"fly\"], {}), ([\"delete\", \"one\"], {}), ([\"--format\", \"xml\", \"list\"], {})):\n        err = io.StringIO()\n        assert main(argv, {}, io.StringIO(), err, env) == 2, argv\n        assert err.getvalue().startswith(\"notes: \"), argv\n    err = io.StringIO()\n    assert main([\"list\"], {}, io.StringIO(), err, {\"NOTES_FORMAT\": \"yaml\"}) == 2\n    assert err.getvalue() == \"notes: NOTES_FORMAT must be text or json\\n\"")

## Warm-up

`parse_args` takes a list of strings, exactly what the shell hands a program in `sys.argv[1:]`. Passing the list yourself, instead of letting it read `sys.argv`, is the trick that makes command-line tools testable, and you'll use it all day.

## argparse in depth

A command line is made of **positional arguments** (identified by where they are: `cp SOURCE DEST`) and **options** (identified by a name: `--width 200`, `-v`). `argparse` turns a description of them into a parser that checks the input, converts it, writes the `--help` text and reports mistakes.

| You want | Write |
|---|---|
| a required positional | `add_argument("path")` |
| one or more (or zero or more) | `nargs="+"` (or `nargs="*"`); an optional one: `nargs="?"` |
| a converted value | `type=int`, `type=float`, `type=pathlib.Path`, or any function of one string |
| a fixed set of values | `choices=["png", "jpg"]` |
| a value with a default | `--width`, `default=800` |
| a required option | `required=True` (but if it's required, consider a positional) |
| an on/off flag | `action="store_true"` (`False` unless given) |
| a repeatable flag | `-v`, `action="count", default=0` (`-vvv` gives 3) |
| a repeatable option | `action="append"` (`--tag a --tag b` gives `["a", "b"]`) |
| two options that can't go together | `group = parser.add_mutually_exclusive_group()` then `group.add_argument(...)` |

Each option has a **destination**: `--dry-run` is stored as `args.dry_run` (dashes become underscores), or wherever `dest=` says. Short and long names can share one option: `add_argument("-q", "--quiet", action="store_true")`. Add `help="..."` to everything: `--help` is the first documentation anyone reads, and argparse writes it for free, with a usage line built from what you declared.

A **type** is any function from a string to a value. Raising `argparse.ArgumentTypeError` (or `ValueError`) inside it turns into a proper usage error, so validation lives in one place and `args` only ever holds good values.

## Subcommands

Big tools are families of commands: `git commit`, `git log`, `pip install`. argparse builds them with **subparsers**: each subcommand gets its own parser with its own arguments, and `set_defaults(func=...)` records which function handles it, so `main` just calls `args.func(args)`.

`required=True` makes forgetting the subcommand a usage error instead of an `AttributeError` on `args.func`. Options that every subcommand shares (such as `--verbose` or `--data FILE`) can go on the **main** parser, before the subcommand (`todo --data tasks.json list`), or on a parent parser passed to each subparser with `parents=[common]`.

## Exit codes and streams

A command-line tool talks to the shell, and to any script that runs it, through three channels:

- **stdout**: the **result**, and nothing else, so it can be piped into another program (`todo list | grep urgent`) or saved to a file;
- **stderr**: messages for the **person**: errors, warnings, progress. They still appear on screen when stdout is redirected, and they never pollute the data;
- the **exit code**: `0` means success, anything else means failure. Shells (`&&`, `||`, `set -e`), CI systems, cron and other programs all decide what to do next from it.

The conventions: **`0`** success, **`1`** a general failure (a file was missing, a check found problems), **`2`** a usage mistake (argparse uses `2` when the command line is wrong). Some tools add more (`grep` exits `1` for "no match", which isn't an error at all), and they document them. In Python, `sys.exit(code)` ends the program with that code (it raises `SystemExit`), `sys.exit("message")` prints the message to stderr and exits with `1`, and a crash exits with `1` after a traceback.

When the command line is wrong, argparse prints the usage and the problem **to stderr** and exits with **2**. That's right for a real program but awkward to test, and wrong inside a larger program that only wants to report the problem. Two ways to take control:

- catch it: `try: args = parser.parse_args(argv)` / `except SystemExit as e: return e.code`;
- or subclass the parser and override `error(message)` to raise your own exception, so you decide what happens.

The program's real entry point is then two lines, kept apart from everything testable:

```python
if __name__ == "__main__":
    sys.exit(main(sys.argv[1:]))
```

## Settings and testable tools

A tool's settings usually come from several places, and there's a standard order, **lowest priority first**:

1. built-in **defaults** in the code;
2. a **config file** (an INI file read with `configparser`, a TOML file read with `tomllib`, or JSON), often in the user's home folder or the project;
3. **environment variables** (`TODO_DATA=~/tasks.json`), handy in scripts, containers and CI;
4. **command-line flags**, the most specific, so they win.

Each later source overrides only the settings it actually sets. The catch with argparse: an option you didn't type still has a value (its default), so you can't tell "not given" from "given as the default". Use `default=None` (or `argparse.SUPPRESS`) for options that take part in the merge, and treat `None` as "not given".

Environment variables are always strings: convert them (`"0"`, `"false"` and `"no"` should mean `False`) and report a bad one clearly.

**Testable tools.** Everything above leads to one shape, used by well-made tools everywhere:

- `main(argv=None, ...)` parses `argv` (`None` means `sys.argv[1:]`), does the work and **returns** the exit code; it never calls `sys.exit` itself;
- the output streams, the environment, the clock, the input function and the file locations are **parameters** with sensible defaults, so a test can pass its own;
- the real work lives in ordinary functions that take values and return values; the CLI layer only translates between the command line and them.

Then a test is one line: `assert main(["add", "x"], out=buf) == 0`. Two more habits make tools pleasant: a `--dry-run` that shows what **would** happen without doing it, and a confirmation for anything destructive that a `--yes` flag skips (scripts can't answer prompts).

### On your own computer

Make a tool installable with an **entry point** in `pyproject.toml` (Week 73):

```toml
[project.scripts]
todo = "todo.cli:main"
```

After `pip install -e .`, typing `todo add "Write report"` runs `todo.cli.main()`, which returns the exit code. Two popular libraries build on the same ideas with less code: **Click** (decorators for commands and options) and **Typer** (options taken from type hints, built on Click). Learn argparse first: it's in the standard library, so it's always there.

## Practice

## Quiz

## Daily challenge

## Before you go

Run `git --help`, `pip --help` and `python -m http.server --help` on your own computer and look at how each one arranges its options and subcommands. Tomorrow: **automating files**: finding, renaming, copying, archiving and tidying them with `pathlib`, `shutil` and `zipfile`.